In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import chromadb
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder

C:\Users\WELCome\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\WELCome\AppData\Local\Temp\ipykernel_8268\3105268520.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [3]:
loader=PyMuPDFLoader("Java.pdf")
documents = loader.load()

In [4]:
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=80,separators=["\n\n","\n"," ",""])

In [5]:
chunks=splitter.split_documents(documents=documents)


In [6]:
MODELNAME="all-MiniLM-L6-v2"

In [7]:
model=SentenceTransformer(MODELNAME)

In [8]:
chunksPageList=[]
for i in range(len(chunks)):
    chunksPageList.append(chunks[i].page_content)

In [9]:
embedding=model.encode(chunksPageList)

In [10]:
client=chromadb.PersistentClient(path="hybrid_search")

In [11]:
try:
    collection=client.get_or_create_collection(name="hybridRAGSystem",metadata={"Description" : "Hybrid RAG System"})
    print(f"Vector DB is Created {collection}")
    print(f"Exising Document  {collection.count()}")
    
except Exception as e:
    print(str(e))

Vector DB is Created Collection(name=hybridRAGSystem)
Exising Document  1231


In [12]:

ids=[f"chunks_{i}" for i in range(len(chunks))]


In [13]:

existing_ids = set(collection.get()['ids'])
new_ids, new_docs, new_embs = [], [], []

for i, (doc, emb) in enumerate(zip(chunksPageList, embedding.tolist())):
    cid = f"chunks_{i}"
    if cid not in existing_ids:
        new_ids.append(cid)
        new_docs.append(doc)
        new_embs.append(emb)

if new_ids:
    collection.add(ids=new_ids, documents=new_docs, embeddings=new_embs)
    print(f"Added {len(new_ids)} chunks. Skipped {len(chunksPageList)-len(new_ids)} duplicates.")
else:
    print("All chunks already indexed. Nothing added.")


All chunks already indexed. Nothing added.


In [14]:
def toknizedSentence(doc):
    return [query.lower().split() for query in doc]

In [15]:
tokenized=toknizedSentence(chunksPageList)


In [16]:
bm25=BM25Okapi(tokenized)


In [21]:
def hybrid_search(query,top_k=3,alpha=0.4):
    beta=1-alpha

    # BM25 Tokenized Query and get Score
    tokenized_query = query.lower().split()
    bm25_score=bm25.get_scores(tokenized_query)


    # print(f"Key word Seach{keyword_search_scores}")
    # print(f"Vector word Seach{vector_search_query}")

    # Query Embedding and its Cosine Similarity Score

    query_embedding = model.encode([query])
    vector_score=cosine_similarity(query_embedding,embedding)[0]

    # We have to normalize the scoe of both BM25 and Vector score because their score are in different scale 
    # like BM25 can range like 1.223,2.32 and Vector can score in range between -1 to 1 that's why after normalize we can combine its result and get the actual document.

    # ---------------Normalize BM25 Score----------------------
    bm25_max=np.max(bm25_score)
    bm25_min=np.min(bm25_score)


    if bm25_max-bm25_min == 0:
        bm25_normalized=np.zeros(len(bm25_score))
    else:
        bm25_normalized=((bm25_score-bm25_min)/(bm25_max-bm25_min))

    # ---------------Normalize Vectore Score----------------------
    vector_max=np.max(vector_score)
    vector_min=np.min(vector_score)


    if vector_max-vector_min == 0:
        vec_normalized=np.zeros(len(vector_score))
    else:
        vec_normalized=((vector_score-vector_min)/(vector_max-vector_min))

    # Final Score like this
    #  alpha=0.5 means 50% vector + 50% BM25
    # alpha=1.0 means pure vector, alpha=0.0 means pure BM25

    final_score=(alpha * vec_normalized + beta * bm25_normalized)

    ranked_index=np.argsort(final_score)[::-1]

    result=[]

    for idx in ranked_index[:top_k]:
        result.append({
            "documents" : chunksPageList[idx],
            "bm25score" : bm25_score[idx],
            "vectorscore" : vector_score[idx],
            "finalScore" : final_score[idx]
        })

    # print(vec_normalized)
    return result

    

In [22]:
hybrid_search("Explain Java With Its Features")

[{'documents': 'JAVA PROGRAMMING \nPage 5 \n \nFeatures of Java \nThere is given many features of java. They are also known as java buzzwords. The Java Features \ngiven below are simple and easy to understand. \n1. Simple \n2. Object-Oriented \n3. Portable \n4. Platformindependent \n5. Secured \n6. Robust',
  'bm25score': 8.864283923418988,
  'vectorscore': 0.78397036,
  'finalScore': 1.0000000059604646},
 {'documents': 'JAVA PROGRAMMING \nPage 43 \n \n \n \n \nAbstract class in Java \n \nA class that is declared with abstract keyword is known as abstract class in java. It can have \nabstract and non-abstract methods (method with body). It needs to be extended and its method \nimplemented. It cannot be instantiated.',
  'bm25score': 6.808450835441059,
  'vectorscore': 0.4743063,
  'finalScore': 0.7090612788722015},
 {'documents': 'Swing is built on the AWT. Two key Swing features are: Swing components are light weight, Swing supports a \npluggable look and feel. \n \n \n \n \n \n \n \n

In [23]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq 

In [24]:
load_dotenv()
groq=os.getenv("GROQ_API_KEY")

In [25]:
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.2,max_tokens=None)

In [26]:
reranker=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [27]:
def rerankScore(query,retrived_docs,top_k=3):
    pair=[(query,doc['documents']) for doc in retrived_docs]
    scores =reranker.predict(pair)
    for doc,rerank_score  in zip(retrived_docs,scores):
        doc['reranker_score'] = float(rerank_score)
    reranked=sorted(
        retrived_docs,
        key=lambda x:x['reranker_score'],
        reverse=True
    )
    return reranked[:top_k]

In [28]:
from langchain_core.messages import SystemMessage,HumanMessage

In [30]:
MULTI_QUERY_PROMPT = """
You are a query expansion assistant for a Hybrid RAG retrieval system.

Generate multiple alternative search queries for the user's question.

Rules:
- Preserve original meaning
- Include technical keywords if relevant
- Make each query different
- Keep them concise and searchable
- Return ONLY the queries
- One query per line
- Do NOT explain
- Do NOT answer
"""
def generateMultipleQuerys(query,num_queries=4):
    message=[
        SystemMessage(content=MULTI_QUERY_PROMPT),
        HumanMessage(content=query)
    ]
    response=llm.invoke(message)
    queries=response.content.strip().split("\n")
    clean_queries=[
        q.strip("- ").strip()
        for q in queries
        if q.strip()
    ]
    return clean_queries[:num_queries]


In [31]:
def multi_query_retrive(queries,fetch_k=5):    
    all_res=[]
    for q in queries:
        res=hybrid_search(query=q,top_k=fetch_k)
        all_res.extend(res)
    return all_res

In [32]:
def duplicate_docs(doc):
    seen=set()
    unique_docs=[]
    for d in doc:
        content = d['documents']
        if content not in seen:
            seen.add(content)
            unique_docs.append(d)
    return unique_docs


In [33]:
RELEVANCE_CHECK_PROMPT = """
You are a retrieval relevance evaluator.

Your task:
Check whether the provided context contains enough relevant information
to answer the user's question.

Rules:
- Return ONLY YES or NO
- YES = context is relevant enough
- NO = context is unrelated, weak, or insufficient
- Do NOT explain
"""
def check_relevance_answer(query,context):
    message=[
        SystemMessage(content=RELEVANCE_CHECK_PROMPT),
        HumanMessage(content=f"Context{context} Question{query}")
    ]
    response=llm.invoke(message)
    decision=response.content.strip().upper()
    if "YES" in decision:
        return True
    return False


In [34]:
SYSTEM_PROMPT = """
Answer using ONLY the provided context.
If no relevant context exists, say:
"The provided documents don't cover this."
"""

# The whole point of a reranker is to over-fetch then narrow down
def ragSystem(query,fetch_k=15,final_k=3,retry_fetch_k=10):
    # rewriteQuery=query_rewrite(query=query)

    # Multiple Query Generator
    multi_queries=generateMultipleQuerys(query=query)
    # print(query)
    # print(multi_queries)
    
    # Retrive Document From The Hybrid Search
    reterived_docs=multi_query_retrive(queries=multi_queries,fetch_k=fetch_k)
    
    if not reterived_docs:
        return "No Relavant Document is Found."

    unique_Docs=duplicate_docs(reterived_docs)
    reranked_docs=rerankScore(query,unique_Docs,top_k=final_k)

    if not reranked_docs:
        return "No relevant documents found."

    # print(reterived_docs)
    context="\n\n-------\n\n".join([doc['documents'] for doc in reranked_docs])
    

    # Relevance Check 
    is_relevant=check_relevance_answer(query,context)
    if not is_relevant:
        print("Retrive Answer is Weak We Are Trying To Fetch Some more Content")
        reterived_docs=multi_query_retrive(queries=multi_queries,fetch_k=retry_fetch_k)
        unique_Docs=duplicate_docs(reterived_docs)
        reranked_docs=rerankScore(query,unique_Docs,top_k=final_k)
        if not reranked_docs:
            return "The provided documents don't cover this."
        context = "\n\n-------\n\n".join(
            [doc["documents"] for doc in reranked_docs]
        )
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Context={context}\n\nQuestion {query} ")
    ]
    response=llm.invoke(messages)
    return response.content


In [52]:
print(ragSystem("What is File Handling in java with Example."))

In Java, File Handling refers to the process of reading and writing data to a file. The two important streams used for file handling are FileInputStream and FileOutputStream. 

FileOutputStream is an output stream used for writing data to a file, and it belongs to byte streams. It can be used to create text files.

For example, FileOutputStream can be used to write data to a file, while FileInputStream can be used to read data from a file.

Here's a simple example:
```java
// Create a FileOutputStream object
FileOutputStream fos = new FileOutputStream("example.txt");

// Write data to the file
fos.write("Hello, World!".getBytes());

// Close the stream
fos.close();
```
This example creates a new file called "example.txt" and writes the string "Hello, World!" to it. 

Similarly, FileInputStream can be used to read data from a file:
```java
// Create a FileInputStream object
FileInputStream fis = new FileInputStream("example.txt");

// Read data from the file
int ch;
while ((ch = fis.rea